## Utilidades

In [991]:
import pandas as pd
import numpy as np


In [992]:
contracts = pd.read_csv("contracts.csv")
customers = pd.read_csv("customers.csv")
usage     = pd.read_csv("usage.csv")

# 1. Descubrimiento y Comprensión de Datos

In [993]:
contracts.head(-10)
#customers.head(-10) 
#usage.head()

,con_id,cli_ref,p_type,val_mo,st,dt_start
0,CON_000001,C_00995,F1,51.48,0,2020-07-26
1,CON_000002,C_00098,M2,112.79,1,2021-11-26
2,CON_000003,C_00506,F1,30.34,1,2020-02-23
3,CON_000004,C_01821,F1,81.40,1,2022-07-18
4,CON_000005,C_01071,F1,77.26,1,2022-11-25
...,...,...,...,...,...,...
5485,CON_005486,C_00458,F1,22.28,1,2021-05-24
5486,CON_005487,C_04582,M1,93.24,1,2021-07-20
5487,CON_005488,C_00428,F2,50.49,1,2022-07-02
5488,CON_005489,C_02416,M2,51.89,1,2020-04-07


### Renombrar columnas

In [994]:
contracts.rename(columns={
    "con_id":   "contract_id", # PK unica = 5500 registros
    "cli_ref":  "customer_id", # FK a customers (3395 coincidencias, 2 se repiten)
    "p_type":   "plan_type", # tipo de plan 
    "val_mo":   "monthly_value", # valor mensual del contrato
    "st":       "status", # estado del contrato (3 valores: )
    "dt_start": "start_date", # fecha de inicio del contrato
}, inplace=True)

customers.rename(columns={
    "id_cli":       "customer_id", # PK unica (revisar, no es unica) = 4994 registros // solución: coincidir entre customer_id y name
    "full_name":    "name", # nombre del cliente
    "contact_info": "email", # Contacto
    "reg_date":     "registration_date", # fecha de registro
    "cat":          "category", # 4 valores : A, B, C, X
    "int_score":    "internal_score", # puntaje interno del cliente (0-100)
}, inplace=True)

usage.rename(columns={
    "u_id":  "usage_id", # PK unica
    "c_ref": "contract_id", # FK contracts
    "qty":   "quantity", # consumo
    "ts":    "timestamp", # fecha y hora del consumo
    "loc":   "location", # ubicacion del consumo
}, inplace=True)

### Exploración

#### Info tablas

In [995]:
customers.info() 
contracts.info()
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        5000 non-null   str    
 1   name               5000 non-null   str    
 2   email              5000 non-null   str    
 3   registration_date  5000 non-null   str    
 4   category           5000 non-null   str    
 5   internal_score     5000 non-null   float64
dtypes: float64(1), str(5)
memory usage: 234.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 5500 entries, 0 to 5499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   contract_id    5500 non-null   str    
 1   customer_id    5500 non-null   str    
 2   plan_type      5500 non-null   str    
 3   monthly_value  5500 non-null   float64
 4   status         5500 non-null   int64  
 5   start_date     5500 non-null   str    
dtypes: float64(1), int6

In [996]:
# columnas del tipo datetime
customers['registration_date'] = pd.to_datetime(customers['registration_date'])
contracts['start_date'] = pd.to_datetime(contracts['start_date'])
usage['timestamp'] = pd.to_datetime(usage['timestamp'])

In [997]:
customer_id_nunique = customers['customer_id'].nunique()
contracts_id_nunique = contracts['contract_id'].nunique()
customer_id_contract_nunique = contracts['customer_id'].nunique()
customer_id_nunique, contracts_id_nunique, customer_id_contract_nunique

(4994, 5500, 3395)

In [998]:
# Cuenta cuántos registros de contracts tienen un customer_id presente en customers y viseversa
coincidencias_C_in_Q = contracts['customer_id'].isin(customers['customer_id']).sum() 
coincidencias_Q_in_C = customers['customer_id'].isin(contracts['customer_id']).sum()
coincidencias_C_in_Q, coincidencias_Q_in_C

(np.int64(5497), np.int64(3399))

In [999]:
# Valores únicos comunes
comunes = len(set(contracts['customer_id']) & set(customers['customer_id']))
comunes  # 3393 coincidencias y 2 se repiten esto es porque un cliente puede 
         #tener más de un contrato, pero un contrato no puede tener más de un cliente. (1:N)

3393

#### Entendiendo inconsistencia customer_id

In [1000]:
filas_duplicadas = customers[customers['customer_id'].duplicated(keep=False)].sort_values(by='registration_date')
filas_duplicadas 

,customer_id,name,email,registration_date,category,internal_score
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
0,C_00001,User_1,user_1@domain.com,2022-08-02,A,41.05
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20


In [1001]:
contract_C_01 = contracts[contracts['customer_id'] == 'C_00001']
contract_C_01

,contract_id,customer_id,plan_type,monthly_value,status,start_date
1838,CON_001839,C_00001,M2,112.97,-1,2020-11-17


In [1002]:
contract_U_01 = usage[usage['contract_id'] == 'CON_001839']
contract_U_01

,usage_id,contract_id,quantity,timestamp,location
8327,U_0008328,CON_001839,15.72,2024-03-12 15:56:16,US
9048,U_0009049,CON_001839,1.29,2024-06-08 04:04:20,UK
10047,U_0010048,CON_001839,0.42,2024-10-20 13:09:40,US
18951,U_0018952,CON_001839,34.59,2024-03-06 22:00:13,ES
20086,U_0020087,CON_001839,2.83,2024-07-05 23:20:39,??
30360,U_0030361,CON_001839,11.77,2024-01-20 18:48:47,DE
31936,U_0031937,CON_001839,1.71,2024-12-02 02:16:53,ES
33495,U_0033496,CON_001839,8.97,2024-10-13 01:13:04,??
42835,U_0042836,CON_001839,2.00,2024-11-25 20:34:00,DE
46541,U_0046542,CON_001839,14.93,2024-04-17 03:23:21,FR


In [1003]:
# Teoria: Si el formato de customer_id es consistente, debería ser algo como 'C_00001', 'C_00002', etc. Esto implica que el número después de 'C_' 
# debería coincidir entre contracts y customers para el mismo cliente.
# Uso .extract para string con regex hecha con IA (busca 1 o más dígitos) y luego convierto a int para comparar numéricamente.
id_num = customers['customer_id'].str.extract(r'(\d+)').astype(int)
name_num = customers['name'].str.extract(r'(\d+)').astype(int)

# Filtrar las filas donde NO coinciden (patrón se rompe)
no_coinciden = customers[id_num[0] != name_num[0]]
no_coinciden

# Entiendo que cada customer_id tiene coincidencia por name y en este caso de trata de 6 clientes que tenian mal asignado el custormer_id ("conclusión")

,customer_id,name,email,registration_date,category,internal_score
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61


#### Entendiendo inconsistencia Email

In [1004]:
# Extraer el número de la columna 'name' (ej: '4987')
num_name = customers['name'].str.extract(r'(\d+)')[0]

# Construir el email que debería tener según el patrón
email_esperado = "user_" + num_name + "@domain.com"

# Identificar las filas donde el email real no coincide
errores = customers['email'] != email_esperado

# Filtrar y contar
emails_incorrectos = customers[errores]
cantidad_errores = len(emails_incorrectos)

cantidad_errores, emails_incorrectos[['name', 'email', 'customer_id']]

(500,
            name         email customer_id
 9       User_10    invalid_10     C_00010
 19      User_20    invalid_20     C_00020
 29      User_30    invalid_30     C_00030
 39      User_40    invalid_40     C_00040
 49      User_50    invalid_50     C_00050
 ...         ...           ...         ...
 4959  User_4960  invalid_4960     C_04960
 4969  User_4970  invalid_4970     C_04970
 4979  User_4980  invalid_4980     C_04980
 4989  User_4990  invalid_4990     C_04990
 4999  User_5000  invalid_5000     C_05000
 
 [500 rows x 3 columns])

In [1005]:
# y cuantos coinciden con el patrón invalid?

email_invalid = "invalid_" + num_name
errores_invalid = customers['email'] != email_invalid

cantidad_invalid = len(errores_invalid)
cantidad_invalid

5000

In [1006]:
# Comprobar que no hay inconsistencia entre name e email respecto al id

# Identificar qué filas de la tabla TIENEN un formato "invalid_"
invalid_rows = customers['email'].str.contains('invalid_')

# Extraer el número del email actual para comparar
num_invalid_email = customers['email'].str.extract(r'invalid_(\d+)')[0]

# Comparar numeros: Solo en las filas que son 'invalid_'
coinciden_num = num_name[invalid_rows] == num_invalid_email[invalid_rows]

invalid_rows.sum(), coinciden_num.sum()

(np.int64(500), np.int64(500))

#### Exporación inconsistencia monthly_value

In [1007]:
val_mo = contracts['monthly_value'].describe()
val_mo
#se observa que el valor mínimo es negativo y el máximo es muy alto, lo que sugiere posibles errores en los datos.
# La desviacion estandar es alta, lo que indica una gran variabilidad en los valores mensuales de los contratos.


count    5500.000000
mean       88.058615
std       444.796588
min        -1.000000
25%        42.277500
50%        68.725000
75%        94.470000
max      9999.990000
Name: monthly_value, dtype: float64

In [1008]:
valor_menor_0 = contracts[contracts["monthly_value"] < 0]
valor_menor_0

,contract_id,customer_id,plan_type,monthly_value,status,start_date
111,CON_000112,C_04018,M2,-1.0,-1,2021-02-05
112,CON_000113,C_00311,M2,-1.0,1,2020-02-14
113,CON_000114,C_00740,F2,-1.0,1,2020-12-26
114,CON_000115,C_00461,M2,-1.0,1,2021-06-20
115,CON_000116,C_01906,F2,-1.0,1,2020-07-26
116,CON_000117,C_04929,F2,-1.0,0,2020-02-12
117,CON_000118,C_01793,M1,-1.0,1,2020-10-08
118,CON_000119,C_00164,F1,-1.0,1,2022-02-15
119,CON_000120,C_04313,F2,-1.0,1,2020-08-15
120,CON_000121,C_03916,F2,-1.0,1,2021-11-24


In [1009]:
valor_mayor_9999 = contracts[contracts["monthly_value"] >= 9999]
valor_mayor_9999

,contract_id,customer_id,plan_type,monthly_value,status,start_date
100,CON_000101,C_00232,F2,9999.99,1,2021-02-11
101,CON_000102,C_02881,F1,9999.99,1,2021-02-09
102,CON_000103,C_04651,F2,9999.99,1,2021-08-04
103,CON_000104,C_04618,M2,9999.99,1,2020-12-22
104,CON_000105,C_01165,M2,9999.99,1,2021-03-24
105,CON_000106,C_04613,M2,9999.99,1,2021-06-20
106,CON_000107,C_01543,F1,9999.99,1,2020-12-22
107,CON_000108,C_04033,F2,9999.99,-1,2021-11-26
108,CON_000109,C_04442,F2,9999.99,0,2023-02-09
109,CON_000110,C_02532,M1,9999.99,1,2021-05-18


In [1010]:
# Unir usage con contracts (por contract_id) y luego con customers (por customer_id)    
df_tres_tablas = usage.merge(contracts, on='contract_id').merge(customers, on='customer_id')

In [1011]:
# Calcular matriz de correlación
matriz_corr = df_tres_tablas.corr(numeric_only=True)

# Ver la correlación de una columna
print(matriz_corr['monthly_value'])

quantity          0.000264
monthly_value     1.000000
status            0.000185
internal_score   -0.006212
Name: monthly_value, dtype: float64


In [1012]:
# Analizar correlación aunque ya veo que no hay mucha relación.
contract_usage = contracts[contracts['contract_id'] == 'CON_000136']
customer_Contracts = customers[customers['customer_id'] == 'C_02990']
usage_Contracts = usage[usage['contract_id'] == 'CON_000136']
contract_usage, customer_Contracts, usage_Contracts

(    contract_id customer_id plan_type  monthly_value  status start_date
 135  CON_000136     C_02990        F2         105.79       1 2021-01-27,
      customer_id       name         email registration_date category  \
 2989     C_02990  User_2990  invalid_2990        2022-02-16        C   
 
       internal_score  
 2989            62.8  ,
         usage_id contract_id  quantity           timestamp location
 23427  U_0023428  CON_000136      0.56 2024-06-10 19:45:53       DE
 26404  U_0026405  CON_000136      4.81 2024-09-21 16:53:18       DE
 32905  U_0032906  CON_000136      6.47 2024-04-13 03:06:26       ??
 49994  U_0049995  CON_000136      3.31 2024-03-02 10:42:19       UK
 51801  U_0051802  CON_000136      7.01 2024-04-14 13:41:54       DE
 54719  U_0054720  CON_000136      0.01 2024-05-14 05:54:24       UK
 58918  U_0058919  CON_000136      4.38 2024-10-26 08:55:58       UK
 59681  U_0059682  CON_000136      8.04 2024-12-13 01:25:53       ES)

In [1013]:

#Explorar .describe() sin outliers
p01 = contracts["monthly_value"].quantile(0.002)
p99 = contracts["monthly_value"].quantile(0.998)
clean = contracts[contracts["monthly_value"].between(p01, p99)]
print(clean["monthly_value"].describe())

count    5479.000000
mean       68.321316
std        30.272708
min        15.030000
25%        42.495000
50%        68.710000
75%        94.345000
max       119.990000
Name: monthly_value, dtype: float64


#### Explorando Status del contrato

In [1014]:
contracts.head()

,contract_id,customer_id,plan_type,monthly_value,status,start_date
0,CON_000001,C_00995,F1,51.48,0,2020-07-26
1,CON_000002,C_00098,M2,112.79,1,2021-11-26
2,CON_000003,C_00506,F1,30.34,1,2020-02-23
3,CON_000004,C_01821,F1,81.40,1,2022-07-18
4,CON_000005,C_01071,F1,77.26,1,2022-11-25


In [1015]:
# Ver la correlación de una columna (reutilizo)
print(matriz_corr['status'])

quantity          0.002819
monthly_value     0.000185
status            1.000000
internal_score    0.001184
Name: status, dtype: float64


In [1016]:
# Hipótesis: que si un status es -1 = cancelado, no hya consumo reciente
# Unir usage con contracts (necesito status y contract_id)
hipotesis_status1 = usage.merge(contracts[['contract_id', 'status']], on='contract_id', how='left')

In [1017]:
# Agrupar por status y ver la fecha del último consumo
resultado_hipotesis_status = hipotesis_status1.groupby('status')['timestamp'].agg(['max', 'min', 'count'])
resultado_hipotesis_status


,max,min,count
status,,,
-1.0,2024-12-30 21:49:23,2024-01-01 11:25:29,6335
0.0,2024-12-30 22:23:09,2024-01-01 00:56:08,5773
1.0,2024-12-30 23:48:58,2024-01-01 00:03:42,47391


##### Descarto hipótesis, por el momento tomaré la interpretación más típica ya que no tengo datos a día de hoy y no tengo evidancias que me indiquen lo contrario.

#### Exploro tabla usage columna c_ref

In [1018]:
# Que datos de usage tienen coincidencias en contracts en la columna contract_id
contract_id_coinciden = usage['contract_id'].isin(contracts['contract_id'])

# Filas que SÍ tienen coincidencia
usage_con_contrato = usage[contract_id_coinciden]

# Filas que NO tienen coincidencia (el símbolo ~ invierte el filtro)
usage_sin_contrato = usage[~contract_id_coinciden]

print(len(usage_con_contrato), len(usage_sin_contrato))

print(usage_sin_contrato)

59499 501
      usage_id contract_id  quantity           timestamp location
0    U_0000001  CON_999999     24.65 2024-06-01 15:33:49       UK
1    U_0000002  CON_999999      1.81 2024-07-20 01:49:33       UK
2    U_0000003  CON_999999     13.21 2024-08-18 16:51:04       US
3    U_0000004  CON_999999      3.68 2024-08-17 18:10:10       US
4    U_0000005  CON_999999     33.96 2024-10-09 14:13:46       DE
..         ...         ...       ...                 ...      ...
496  U_0000497  CON_999999      3.67 2024-10-12 15:29:11       ES
497  U_0000498  CON_999999     10.63 2024-01-27 16:16:43       UK
498  U_0000499  CON_999999      7.40 2024-07-13 14:56:19       ??
499  U_0000500  CON_999999      8.48 2024-06-21 16:19:18       US
500  U_0000501  CON_999999      1.89 2024-06-30 02:42:40       US

[501 rows x 5 columns]


In [1019]:
# cuantos contract_id son igual a CON_999999     
contratos_con_id_999999 = usage[usage['contract_id'] == 'CON_999999']
print(len(contratos_con_id_999999)) 
total_registros = len(usage)
print(total_registros)
contratos_id_999999 = (usage['contract_id'] == 'CON_999999').sum()
print(contratos_id_999999)

501
60000
501


In [1020]:
# Saber cuanto me represnta ese contrato en porcentaje sobre el total de registros, para evaluar si eliminarlo o no
porcentaje = (contratos_id_999999 / total_registros) * 100
porcentaje


np.float64(0.835)

#### 

#### Anomalias de cantidad de uso

In [1021]:
print(matriz_corr['quantity'])

quantity          1.000000
monthly_value     0.000264
status            0.002819
internal_score    0.007061
Name: quantity, dtype: float64


In [1022]:
usage['quantity'].describe()

count    60000.000000
mean         8.195222
std         20.107887
min         -5.000000
25%          2.310000
50%          5.560000
75%         11.080000
max       1440.000000
Name: quantity, dtype: float64

In [1023]:
usage.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   usage_id     60000 non-null  str           
 1   contract_id  60000 non-null  str           
 2   quantity     60000 non-null  float64       
 3   timestamp    60000 non-null  datetime64[us]
 4   location     60000 non-null  str           
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 2.3 MB


In [1024]:
neg  = usage[usage["quantity"] < 0]
valor_alto = usage[usage["quantity"] >= 1440]

len(neg), len(valor_alto)

#pocos consumos negativos y pocos consumos grandes, lo que sugiere que podrían ser errores o casos atípicos.

(11, 10)

In [1025]:
q_clean = usage[usage["quantity"].between(0.01, 1439)]
q_clean["quantity"].describe()
# se confirma con el .describe() ya que no hay cambios significativos. Creo que el valor negativo 
# si es consumo pueden ser devoluciones y el consumo alto ser atipico pero real

count    59942.000000
mean         7.963837
std          7.910600
min          0.010000
25%          2.320000
50%          5.560000
75%         11.080000
max         83.910000
Name: quantity, dtype: float64

In [1026]:
# Sabes si hay relacion con la variable location
valor_alto["location"].value_counts()

location
US    4
DE    3
ES    1
??    1
FR    1
Name: count, dtype: int64

In [1027]:
neg["location"].value_counts()

location
US    5
UK    3
ES    1
DE    1
FR    1
Name: count, dtype: int64

# 2. Detección y Resolución de Anomalías Contextuales

##### Según lo explorado anteriormente llego a la conclusión de que la BBDD, en la tabla Customers, tiene un patrón entre las columnas customer_id y name y por lo tanto, al tener datos de email y user diferentes llego a la conclusión que son clientres diferentes y se les a asignado mal su customer_id. Corrijo.

In [1028]:
#Solución para inconsistencia de id_cli y full_name

# Extraer el número de la columna 'name' (ej: de 'User_4996' extrae '4996')
numero_name = customers['name'].str.extract(r'(\d+)')[0]

# Convertir a entero y luego a string con formato 'C_' seguido de 5 dígitos (rellenando con ceros)
# El formato :05d asegura que siempre tenga 5 dígitos (04996)
customers['customer_id'] = numero_name.apply(lambda x: f"C_{int(x):05d}")

Solución = customers[customers['customer_id'] == 'C_04996']
customer_id_nunique2 = customers['customer_id'].nunique()
Solución
customer_id_nunique2

5000

##### Como hemos visto hay errores de email con el patrón invalid_(num_name), todos los errores coinciden, se estable patrón correcto que es user_(num_name)@domain.com
##### Se descarta que hayan otros tipos de inconsistencia y que el user de email coincide con el de name, anteriormente vimos que el de name coincide con el de customer.

In [1029]:
# Aplicar el cambio solo a las filas detectadas
customers.loc[errores, 'email'] = "user_" + num_name[errores] + "@domain.com"

email_ok=len(customers[customers['email'] != email_esperado])
email_ok, customers[['name', 'email']]

(0,
            name                 email
 0        User_1     user_1@domain.com
 1        User_2     user_2@domain.com
 2        User_3     user_3@domain.com
 3        User_4     user_4@domain.com
 4        User_5     user_5@domain.com
 ...         ...                   ...
 4995  User_4996  user_4996@domain.com
 4996  User_4997  user_4997@domain.com
 4997  User_4998  user_4998@domain.com
 4998  User_4999  user_4999@domain.com
 4999  User_5000  user_5000@domain.com
 
 [5000 rows x 2 columns])

##### Para la interpretación y validación de val_mo, entiendo que es el valor mensual del contrato, cuando veo del desglose del .describe() veo que hay valores atipicos que no tienen sentido, al hacer el describe excluyendo estos valores veo que me da un resultado lógico. Los valores atipicos son 21 filas de un total de 5500 filas, no es un valor muy alto del total. Opto por considerar estos valores como nulos ya que considero que se han introducido mal.

In [1030]:
# Definir la condición: valores menores a 0 O mayores/iguales a 9999
condicion = (contracts['monthly_value'] < 0) | (contracts['monthly_value'] >= 9999)

# Aplicar la máscara: si se cumple la condición, pone NaN
contracts['monthly_value'] = contracts['monthly_value'].mask(condicion, np.nan)

# Verificar cuántos nulos se crearon
num_nulos_val= contracts['monthly_value'].isna().sum()
num_nulos_val

np.int64(21)

##### Creo una nueva columna con el status_name, tomo la decisión de usar la interpretación típica ya que se trata de un contrato que puede tener estados, no hay evidencias que sea este relacionado con otra variable, descarto la hipóteisis de que pueda estar relacionado con timestamp de tabla Usage. Confirmo que la mayoria de los contratos son de tipo 1 = activo, lo cual tiene sentido en la mayoria de los casos


In [1031]:
# Dicc de mapeo
diccionario_status = {
    1: 'activo',
    0: 'inactivo',
    -1: 'cancelado'
}

# Creo una nueva columna con los nombres de status
contracts['status_name'] = contracts['status'].map(diccionario_status)

# Ver cuántos hay de cada uno
cantidad_contratos= contracts['status_name'].value_counts()

cantidad_contratos, contracts.head()

(status_name
 activo       4386
 cancelado     586
 inactivo      528
 Name: count, dtype: int64,
   contract_id customer_id plan_type  monthly_value  status start_date  \
 0  CON_000001     C_00995        F1          51.48       0 2020-07-26   
 1  CON_000002     C_00098        M2         112.79       1 2021-11-26   
 2  CON_000003     C_00506        F1          30.34       1 2020-02-23   
 3  CON_000004     C_01821        F1          81.40       1 2022-07-18   
 4  CON_000005     C_01071        F1          77.26       1 2022-11-25   
 
   status_name  
 0    inactivo  
 1      activo  
 2      activo  
 3      activo  
 4      activo  )

##### El unico datos que no tiene coincidencia con la tabla de contracts es el CON_999999, veo que son 501 registros. Por lo tanto tengo dos opciones, elimino en uso de este contrato, los 501 registros (menos del 1% del total) o creo un registro de este contrato, evaluo que los datos que me genera la tabla usage no son suficientes para tener un registro completo del contrato, sin tipo de plan, valor mensual, status y fecha de inicio por lo tanto mi elección es eliminar los registros de usage.

In [1032]:
# Elimino todos los registros de usage que su contract_id es igual a CON_999999
usage = usage[usage['contract_id'] != 'CON_999999']
print(len(usage))

59499


##### He tomado la decision de eliminar valores negetivos ya que pueden significar devolución pero como no estoy segura que eso sea posible optaré por eliminarlos ya que es consumo negativo, pero los valores mas altos los dejaré ya que son pocos y es tipico en consumo que exista la posibilidad de alguna vez consumir mucho. No es algo común pero puede pasar y la empresa debe considerar que pueden aparecer outliers que no son errores.

In [1033]:
usage = usage[usage['quantity'] != -5]
print(len(usage))

59488


# 3. Detección de Inconsistencias Lógicas y de Negocio Cruzadas 

#### 1. Fechas de Contrato vs. Registro de Cliente:  Identifica contratos que comenzaron antes de que el cliente se registrara.
Existen casi tantos registros que tienen el "error" de tener la fecha de registro posterior a la fecha de inicio del contrato. Hipotesis, el cliente tiene la posibilidad de hacer el registro independientemente de la contratación (el valor de registration_date no tiene relación con la contratación)
#### ¿Qué implicaciones tiene esta anomalía para la trazabilidad del cliente y la facturación? ¿Cómo la corregirías o marcaría? : 
Si lo tomo como valores independientes no tiene implicancia en la trazabilidad ni facturación. Las opciones para corregirlo pueden ser aplicar la fecha de start_date como fecha de registration_date en los casos donde registration_date sea mayor pero lo veo una medida apresurada teniendo en cuenta que este patron ocurre en la mitad de los datos. Buscaría otra explicación porque dudo que la mitad de los registros esten erroneos.

In [1034]:
#Me traigo la columna registration_date de customers a contracts para comparar con start_date
registration_date_customers = contracts.merge(customers[['customer_id', 'registration_date']])

# Creamos la condición lógica
condicion_error = registration_date_customers['start_date'] < registration_date_customers['registration_date']

# Extraemos los contratos que cumplen el error
contratos_pre_registro = registration_date_customers[condicion_error]


print(len(contratos_pre_registro))
print(contratos_pre_registro[['contract_id', 'customer_id', 'status','start_date', 'registration_date']].head())

2538
  contract_id customer_id  status start_date registration_date
0  CON_000001     C_00995       0 2020-07-26        2021-06-06
1  CON_000002     C_00098       1 2021-11-26        2022-10-04
2  CON_000003     C_00506       1 2020-02-23        2023-09-29
5  CON_000006     C_01624       1 2021-11-03        2022-09-12
6  CON_000007     C_03282       1 2021-01-29        2023-08-21


#### 2. Clientes sin Actividad: Identifica clientes que, según los datos, no tienen contratos activos o no han generado ningún uso en un período reciente. ¿Podrían ser clientes fantasmas o son datos erróneos? ¿Cómo los identificarías y qué acción propondría?

Considero que 3 escenarios, el clientes que ya se fue (2044 registros), el cliente que ya se fue y que estuvo sin usar el servicio por 30 dias 168, y el cliente activo (que tiene contrato) y lleva 90 dias sin usar el servicio. Considero este el riesgo y el resto como dato referente. Ya que el cliente que lleva mucho tiempo sin uso de servicio es una posible baja proxima.
No considero que sean datos erroneos ya que no existe cliente activo que lleve sin actividad periodos muy largos (365 días). A los clientes identificados en riesgo propondria seguimiento y acciones para no perder clientes (desconozco el producto exacto pero con acciones de marketing podría funcionar)

In [1035]:
# la fecha maxima es 30-12-2024 y de ahi calculo en este caso 90 dias menos.
fecha_corte = usage['timestamp'].max() - pd.Timedelta(days=90)
fecha_corte2 = usage['timestamp'].max() - pd.Timedelta(days=30)
# quienes tienen contratos activos
contrato_activos = contracts[contracts["status_name"] == "activo"]["customer_id"]

# Obtener la fecha del ultimo uso por cliente
# merge entre usage y contracts para saber de quién es cada uso, y busco el máximo
ultimo_uso = usage.merge(contracts[['contract_id', 'customer_id']], on='contract_id').groupby('customer_id')['timestamp'].max()

# Aplicar todo a la tabla customers
df_clientes_sin_actividad = customers.copy()
df_clientes_sin_actividad['tiene_contrato_activo'] = df_clientes_sin_actividad['customer_id'].isin(contrato_activos)
df_clientes_sin_actividad['ultimo_uso'] = df_clientes_sin_actividad['customer_id'].map(ultimo_uso) #busca del customer_id el ultimo_uso y lo trae (1:1)

# Filtrar los clientes que están "En Riesgo" (tienen contrato activo y sin uso reciente 90 dias sin uso)
en_riesgo = df_clientes_sin_actividad[
    (df_clientes_sin_actividad['tiene_contrato_activo']) & (df_clientes_sin_actividad['ultimo_uso'] < fecha_corte)
]
#Clientes que no tienen contrato activo
clientes_sin_contrato_activo = df_clientes_sin_actividad[~df_clientes_sin_actividad['tiene_contrato_activo']]

clientes_sin_contrato_activo_en_riesgo = clientes_sin_contrato_activo[clientes_sin_contrato_activo['ultimo_uso'] < fecha_corte2]

print(len(en_riesgo), len(clientes_sin_contrato_activo), len(clientes_sin_contrato_activo_en_riesgo))
en_riesgo

113 2044 168


,customer_id,name,email,registration_date,category,internal_score,tiene_contrato_activo,ultimo_uso
37,C_00038,User_38,user_38@domain.com,2019-01-14,A,34.26,True,2024-09-05 06:12:04
113,C_00114,User_114,user_114@domain.com,2022-08-03,C,53.10,True,2024-09-24 15:42:49
116,C_00117,User_117,user_117@domain.com,2020-03-25,C,64.43,True,2024-08-25 01:43:34
120,C_00121,User_121,user_121@domain.com,2023-04-23,C,35.74,True,2024-09-05 18:36:57
191,C_00192,User_192,user_192@domain.com,2020-11-27,C,69.08,True,2024-09-21 06:50:24
...,...,...,...,...,...,...,...,...
4820,C_04821,User_4821,user_4821@domain.com,2021-10-25,A,32.19,True,2024-08-23 05:48:09
4833,C_04834,User_4834,user_4834@domain.com,2020-08-01,A,29.34,True,2024-09-22 02:30:59
4851,C_04852,User_4852,user_4852@domain.com,2021-08-08,A,45.86,True,2024-08-06 23:46:47
4880,C_04881,User_4881,user_4881@domain.com,2023-10-05,A,53.98,True,2024-09-19 22:16:17


#### 3 Contratos con val_mo atípico: Analiza si los contratos con val_mo que identificaste como atípicos en el punto 2 tienen alguna característica común. ¿Indica algo o es un error? 
Anteriormente identifiqué val_mo como el valor mensual donde los valores atipicos son -1 y 9999.99, considero estos valores como nulos ya que representar menos del 1% de los datos totales de la tabla y no se relacionan con los demas valores.


# 4. Análisis de Negocio Estratégico y KPIs Avanzados  

#### 1. ARPU (Average Revenue Per User) por Categoría de Cliente y Tipo de Plan: Calcula el ingreso promedio por cliente activo, segmentado por categoría de cliente y tipo de plan. ¿Qué segmentos o planes son los más rentables? ¿Hay oportunidades de upselling o cross-selling? 

Los planes mas rentables son llos categoria C plan M1 y M2. Intento obtener información de correlación entre category y plan_type, para detectar upselling o cross-selling pero no veo correlacion entre el precio, categoria y tipo de plan. Es raro, es posible que se deba a que es un dataset sintetico, pero sin relación no puedo determinar si se pueden mejorar estas ventas.

In [1036]:
# contratos activos
contratos_activos = contracts[contracts['status_name'] == 'activo']

# join con tabla customers para tener category
arpu = contratos_activos.merge(customers[['customer_id', 'category']], on='customer_id')

# agrupo por Categoría y Tipo de Plan
arpu_segmentado = arpu.groupby(['category', 'plan_type'])['monthly_value'].mean().reset_index()

# ordeno para detectar el arpu de mayor a menor
arpu_segmentado = arpu_segmentado.sort_values(by='monthly_value', ascending=False)

# cambiar el nombre de columna monthly_value por arpu
arpu_segmentado = arpu_segmentado.rename(columns={'monthly_value': 'ARPU'})

arpu_segmentado

,category,plan_type,ARPU
10,C,M1,74.087570
11,C,M2,73.709804
2,A,M1,70.268549
6,B,M1,70.242881
1,A,F2,70.069880
14,X,M1,69.043241
13,X,F2,68.994692
3,A,M2,67.648090
7,B,M2,67.572174
5,B,F2,66.865000


In [1037]:
# Buscando alguna correlación entre category y plan_type, creo variables dummy para las dos columnas
#df_tres_tablas es la tabla que tiene toda la información unida usada en la corr (reutilizo variable)
corr_analisis = pd.get_dummies(df_tres_tablas, columns=['category', 'plan_type'])

# Correlación enfocada en monthly_value
correlaciones = corr_analisis.corr(numeric_only=True)['monthly_value'].sort_values(ascending=False)
correlaciones

monthly_value     1.000000
category_C        0.026677
plan_type_F2      0.013718
category_A        0.008515
plan_type_M2      0.001688
plan_type_F1      0.000889
quantity          0.000264
status            0.000185
internal_score   -0.006212
category_X       -0.013988
plan_type_M1     -0.016315
category_B       -0.020416
Name: monthly_value, dtype: float64

In [1038]:
corr_analisis

,usage_id,contract_id,quantity,timestamp,location,customer_id,monthly_value,status,start_date,name,...,registration_date,internal_score,category_A,category_B,category_C,category_X,plan_type_F1,plan_type_F2,plan_type_M1,plan_type_M2
0,U_0000502,CON_004660,3.15,2024-04-08 15:59:23,ES,C_02446,105.51,1,2021-05-12,User_2446,...,2023-06-09,28.67,True,False,False,False,True,False,False,False
1,U_0000503,CON_000117,0.17,2024-08-12 10:12:31,UK,C_04929,-1.00,0,2020-02-12,User_4929,...,2023-02-15,65.87,False,True,False,False,False,True,False,False
2,U_0000504,CON_001252,0.92,2024-07-12 03:51:38,ES,C_00742,108.71,1,2020-01-29,User_742,...,2023-01-11,39.51,True,False,False,False,False,True,False,False
3,U_0000505,CON_002797,2.46,2024-07-18 14:54:17,US,C_02610,37.44,0,2021-02-12,User_2610,...,2021-11-29,67.35,True,False,False,False,True,False,False,False
4,U_0000506,CON_002055,0.54,2024-12-19 08:50:50,ES,C_00877,108.32,0,2020-10-17,User_877,...,2022-06-28,59.41,False,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59531,U_0059996,CON_003039,8.31,2024-01-09 17:34:33,US,C_00089,93.33,1,2023-01-24,User_89,...,2020-07-23,62.93,False,False,True,False,False,False,True,False
59532,U_0059997,CON_002183,11.22,2024-08-14 04:28:47,DE,C_04000,20.42,1,2021-04-03,User_4000,...,2022-05-02,54.32,True,False,False,False,False,False,True,False
59533,U_0059998,CON_002509,4.71,2024-01-05 20:00:39,UK,C_03717,88.35,0,2022-09-15,User_3717,...,2023-09-22,61.40,True,False,False,False,True,False,False,False
59534,U_0059999,CON_000004,31.54,2024-12-06 02:48:49,FR,C_01821,81.40,1,2022-07-18,User_1821,...,2020-05-02,35.28,True,False,False,False,True,False,False,False


#### 2. Patrones de Consumo por Ubicación y Plan: Analiza la cantidad de uso promedio y total por ubicación de uso y tipo de plan. ¿Existen diferencias significativas en el consumo internacional o entre planes? ¿Cómo podría la compañía optimizar sus ofertas o infraestructura? 

No existen diferencias demasiado significativas, de media DE (Alemania) se posiciona como la region de mas consumo con poca diferencia frente US (Unite State), el tipo de plan que mas de consume son F1 seguido por M1. 

In [1039]:
# Merge de tablas para tener 'location', 'plan_type' y 'quantity' en un solo sitio
df_uso_planes = usage.merge(contracts[['contract_id', 'plan_type']], on='contract_id')

# promedio y el total
patrones_de_consumo = df_uso_planes.groupby(['location', 'plan_type'])['quantity'].agg(['mean', 'sum']).reset_index()

# Ordeno por total de uso para ver dónde se concentra el tráfico
patrones_de_consumo = patrones_de_consumo.sort_values(by='sum', ascending=False)

# En funcion del lugar la media de mas uso
location_consumo = patrones_de_consumo.groupby('location')['sum'].mean().sort_values(ascending=False)

# En funcion del tipo de plan la media de mas uso
plan_consumo = patrones_de_consumo.groupby('plan_type')['sum'].mean().sort_values(ascending=False)

In [1040]:
print(patrones_de_consumo)

   location plan_type      mean       sum
20       US        F1  8.750058  22645.15
4        DE        F1  8.816428  21891.19
12       FR        F1  8.569640  21664.05
21       US        F2  8.290126  21662.10
6        DE        M1  8.437521  21540.99
10       ES        M1  8.569527  21398.11
5        DE        F2  8.287422  21182.65
0        ??        F1  8.556587  21057.76
23       US        M2  8.691958  20599.94
14       FR        M1  8.206042  20588.96
7        DE        M2  8.559941  20483.94
18       UK        M1  8.231997  20407.12
8        ES        F1  7.964808  20326.19
16       UK        F1  8.087078  20177.26
2        ??        M1  8.047972  20079.69
13       FR        F2  7.879561  19564.95
1        ??        F2  7.925878  19537.29
9        ES        F2  7.784773  19524.21
22       US        M1  7.810798  19386.40
17       UK        F2  7.617740  19310.97
3        ??        M2  8.290237  19233.35
15       FR        M2  7.892694  18981.93
19       UK        M2  7.840063  1

In [1041]:
print(location_consumo)

location
DE    21274.6925
US    21073.3975
FR    20199.9725
??    19977.0225
ES    19772.8450
UK    19668.0750
Name: sum, dtype: float64


In [1042]:
print(plan_consumo)

plan_type
F1    21293.600000
M1    20566.878333
F2    20130.361667
M2    19319.830000
Name: sum, dtype: float64


#### 3. Tasa de Churn (Abandono) por p_type y cat: Calcula la tasa de contratos en estado Terminated o Suspended (churn) por p_type y cat . ¿Qué combinaciones de plan/categoría muestran mayor propensión al churn? ¿Qué factores podrían estar influyendo? 

Las combinaciones que demuestran mas proponsión al churn son M2/C ; F1/A ; y M2/A. Factores que podrían ester influyendo, al comparar con el ARPU (obtenido anteriormente) veo que M2/C tenia uno de los ARPU más alto, posible cliente que paga mucho y exige servicio o facilidad para cambiar por mas economico. Tambien vimos anteriormente que los planes M2 son los que menos se usan, eso puede afectar al churn. Tambien he visto que los Planes F1 suelen ser los más economicos, pueden ser ofertas o tener facilidades de salida del servicio.


In [1043]:
# Nuevo diccionario para ajuste a la consigna

diccionario_churn = {
    1: 'Active',
    0: 'Suspended',
    -1: 'Terminated'
}

# vuelvo a aplicar mapeo para distribuir categorias, reemplazo antiguo status_name
contracts['status_name'] = contracts['status'].map(diccionario_churn)

# Columna booleana para identificar el Churn (Terminated o Suspended) TRUE
contracts['is_churn'] = contracts['status_name'].isin(['Terminated', 'Suspended'])

# Join con customers para obtener la categoría (cat)
churn = contracts.merge(customers[['customer_id', 'category']], on='customer_id')

In [1044]:
# Agrupo por tipo de plan  y categoría por la media
reporte_churn = churn.groupby(['plan_type', 'category'])['is_churn'].mean().reset_index()

# a porcentaje
reporte_churn['churn_%'] = (reporte_churn['is_churn'] * 100).round(2)

# Ordeno por la tasa de churn más alta
reporte_churn = reporte_churn.sort_values('churn_%', ascending=False)

In [1045]:
reporte_churn

,plan_type,category,is_churn,churn_%
14,M2,C,0.274648,27.46
0,F1,A,0.222222,22.22
12,M2,A,0.213592,21.36
8,M1,A,0.213333,21.33
5,F2,B,0.212996,21.30
10,M1,C,0.200000,20.00
6,F2,C,0.198630,19.86
1,F1,B,0.195312,19.53
15,M2,X,0.194690,19.47
11,M1,X,0.194030,19.40


In [1046]:
# Suponiendo que tu tabla se llama reporte_churn
top_churn = reporte_churn.nlargest(3, 'churn_%')
top_churn

,plan_type,category,is_churn,churn_%
14,M2,C,0.274648,27.46
0,F1,A,0.222222,22.22
12,M2,A,0.213592,21.36


#### 4. Identificación de Clientes de Alto Valor (LTV Potencial): Identifica un subconjunto de clientes que considere de “alto valor” o con alto potencial de LTV (Lifetime Value). 
Considero clientes de alto valor a clientes Activos, que pagan por encima de la media y con un score alto, en este caso por encima del 80. Identifico 24 clientes de alto valor. Al final considero ver si hay patron por categoria y plan de cliente y parece que los clientes de categoria A estan muy bien valorados con planes M1, M2, y F1.

In [1047]:
#Considero que un cliente de alto valor es un cliente activo, que paga mucho, pero tambien tiene un internal_score alto.

# Ingreso promedio porque busco alguno superior a la media
ingreso_promedio = contracts['monthly_value'].mean()

# join de contracts con customers para tener info de precio y score (0-100)
cliente_valor = contracts.merge(customers[['customer_id', 'internal_score', 'category']], on='customer_id')

# Agrupo subconjunto de " Cliente Alto Valor" / Condición: Activo + Pago superior al promedio + Score alto 
clientes_vip = cliente_valor[
    (cliente_valor['status_name'] == 'Active') & 
    (cliente_valor['monthly_value'] > ingreso_promedio) & 
    (cliente_valor['internal_score'] >= 80)
]

# creo columna de potencial_score combinando valor mensual y score para ordenar por LTV Potencial
clientes_vip['potencial_score'] = clientes_vip['monthly_value'] * clientes_vip['internal_score']
clientes_vip = clientes_vip.sort_values('potencial_score', ascending=False)

print(len(clientes_vip))
print(clientes_vip[['customer_id', 'category', 'plan_type', 'monthly_value', 'internal_score']])

58
     customer_id category plan_type  monthly_value  internal_score
1639     C_04307        A        M2         114.03          100.66
5119     C_02409        X        F2         118.47           91.70
3779     C_01144        A        F1         119.18           84.87
1149     C_04401        A        M2         113.58           88.75
551      C_02409        X        F2         108.32           91.70
4624     C_03931        X        M2         105.50           92.98
2153     C_02261        B        F2         114.26           84.38
294      C_02817        A        M2         118.87           80.96
2344     C_04014        A        M1         116.54           81.49
2017     C_04352        A        F1         114.35           82.86
4635     C_01732        B        M1         100.75           92.36
1340     C_01048        A        M2         107.71           85.88
2674     C_04661        A        M2         103.46           88.79
3980     C_04401        A        M2         101.44         

In [1048]:
# Cuantos clientes de cada categoria hay dentro de los clientes VIP
valor_x_cat = clientes_vip.groupby('category').size().sort_values(ascending=False)
# Cuantos clientes de cada tipo de plan hay dentro de los clientes VIP
valor_x_plan = clientes_vip.groupby('plan_type').size().sort_values(ascending=False)
#Cuantos clientes VIP hay por categoría y tipo de plan
valor_x_cat_plan = clientes_vip.groupby(['category', 'plan_type']).size().unstack(fill_value=0)

valor_x_cat, valor_x_plan, valor_x_cat_plan


(category
 A    38
 B    14
 X     4
 C     2
 dtype: int64,
 plan_type
 M1    20
 M2    18
 F1    13
 F2     7
 dtype: int64,
 plan_type  F1  F2  M1  M2
 category                 
 A          10   3  12  13
 B           1   2   7   4
 C           1   0   1   0
 X           1   2   0   1)

# 5. Modelado de Relaciones y Agregaciones de Negocio 

#### 1. Análisis de Cohorts de Registro: Agrupa a los clientes por su fecha de registro en cohorts mensuales o trimestrales. Analiza la evolución del número de contratos activos y el ARPU para cada cohort a lo largo del tiempo. ¿Qué observas?  .

Creo el pd analisis mensual para la revision, tambien traigo correlación, aunque es bastante evidente que el numero de clientes afecta en los ingresos. 
Es posible que en ocasiones como 12-2023 se obtengan menos clientes pero de un valor mayor generando un ARPU mas alto. Pero en general a simple vista no denoto patrón temporal. 

In [1049]:
# Creo cohorte mensual
customers['cohort_month'] = customers['registration_date'].dt.to_period('M')

In [1050]:
# Unimos para tener el valor mensual de sus contratos, el customer_id, monthly_value y status == activo
df_cohorts = customers.merge(contracts[['customer_id', 'monthly_value', 'status']], on='customer_id').query('status == 1')

# Agrupamos por Cohort Mensual
analisis_mensual = df_cohorts.groupby('cohort_month').agg(
    num_clientes=('customer_id', 'count'),
    revenue_total=('monthly_value', 'sum'),
    ARPU=('monthly_value', 'mean')
).reset_index()

print(analisis_mensual.sort_values('cohort_month'))

   cohort_month  num_clientes  revenue_total       ARPU
0       2019-01            60        4281.72  71.362000
1       2019-02            35        2225.80  63.594286
2       2019-03            81        5899.36  72.831605
3       2019-04            65        4385.14  67.463692
4       2019-05            79        5643.29  71.434051
5       2019-06           101        6791.82  67.918200
6       2019-07            89        6261.89  70.358315
7       2019-08            67        4785.52  71.425672
8       2019-09            63        4304.39  69.425645
9       2019-10            82        5344.44  65.176098
10      2019-11            63        3794.95  60.237302
11      2019-12            56        3743.21  66.843036
12      2020-01            74        4740.27  64.057703
13      2020-02            62        4425.52  72.549508
14      2020-03            59        3858.16  65.392542
15      2020-04            79        5505.95  69.695570
16      2020-05            95        6494.38  68

In [1051]:
correlaciones_cohortes = analisis_mensual.corr(numeric_only=True)['num_clientes'].sort_values(ascending=False)
correlaciones_cohortes

num_clientes     1.000000
revenue_total    0.971150
ARPU             0.050015
Name: num_clientes, dtype: float64

In [1052]:
top_num_clientes = analisis_mensual.nlargest(7, 'num_clientes')
top_num_clientes

,cohort_month,num_clientes,revenue_total,ARPU
5,2019-06,101,6791.82,67.918200
18,2020-07,101,6751.97,66.851188
24,2021-01,96,6602.13,69.496105
16,2020-05,95,6494.38,68.361895
45,2022-10,94,6146.41,65.387340
43,2022-08,91,6394.69,70.271319
46,2022-11,91,6308.95,69.329121


In [1053]:
menor_num_clientes = analisis_mensual.nsmallest(7, 'num_clientes')
menor_num_clientes

,cohort_month,num_clientes,revenue_total,ARPU
59,2023-12,17,1196.85,70.402941
1,2019-02,35,2225.80,63.594286
11,2019-12,56,3743.21,66.843036
23,2020-12,58,4216.84,73.979649
56,2023-09,58,3768.38,64.972069
14,2020-03,59,3858.16,65.392542
0,2019-01,60,4281.72,71.362000


#### 2. Impacto de la Categoría de Cliente en el Churn: Realiza un análisis más profundo para determinar si la categoría de cliente tiene un impacto significativo en la probabilidad de que un contrato entre en estado de churn (Utiliza técnicas estadísticas básicas) 

Reutilizo el df que cree anteriormente con los registros churn, voy a optar por analizar la desviación sobre la media, al ver que la media de churn es 19.55% y que, las categorias A y C estan en el 21% puedo detectar que son las que tengo en riesgo y debo prestar atención, mi categoria mas estable y segura es la X.

In [1054]:
# media de Churn Global (el promedio de todos los clientes)
churn_global = reporte_churn['is_churn'].mean()

# media de Churn por Categoría
churn_cat = reporte_churn.groupby('category')['is_churn'].mean().reset_index()

# Diferencia Relativa / Esto nos dice que porcentaje es más (o menos) probable que alguien se vaya según su categoría
churn_cat['desviacion_vs_global'] = (churn_cat['is_churn'] - churn_global) / churn_global *100

# ordeno
churn_cat = churn_cat.sort_values('desviacion_vs_global', ascending=False)

print(churn_global * 100)
print(churn_cat)

19.55893060737753
  category  is_churn  desviacion_vs_global
0        A  0.210548              7.648185
2        C  0.210304              7.523383
1        B  0.188991             -3.373406
3        X  0.172513            -11.798162


#### 3. Correlación entre val_mo y qty : Investiga si existe una correlación entre la tarifa mensual y la cantidad de uso. ¿Los clientes que pagan más usan más? ¿Hay excepciones que sugieran un mal ajuste del plan?  
La correlacion indica que no existe correlacion entre las variables de tarifa y cantidad de uso. Es decir por pagar mas no se usa mas. También he comprobado la correlacion por tipo de plan pero tampoco hay correlacion, es decir por usar un tipo de plan espeficico de plan no indica que se use mas.

In [1061]:
# join entre usage y contracts para tener quantity y monthly_value en un mismo sitio
val_qua_correlacion = usage.merge(contracts[['contract_id', 'monthly_value', 'plan_type']], on='contract_id')

# calcula correlacion entre quantity y monthly_value
correlacion_val = val_qua_correlacion['quantity'].corr(val_qua_correlacion['monthly_value'])

# calcula xorrelacion entre quantity y plan_type (obtengo variable dummy para plan_type de df_tres_tablas que ya tiene toda la info unida)
plan_type_dummies = pd.get_dummies(val_qua_correlacion['plan_type'])
plan_type_dummies['quantity'] = val_qua_correlacion['quantity']  
correlacion_plan = plan_type_dummies.corr(numeric_only=True)['quantity'].sort_values(ascending=False)

print(correlacion_val)
correlacion_plan


-0.0028948446031553123


quantity    1.000000
F1          0.007402
M1          0.000503
M2         -0.001143
F2         -0.006777
Name: quantity, dtype: float64